# 6. Generator realism and rxncon → GSM transfer

## Goal

Do the conclusions survive a richer causal chain—realized process variation, regulatory/rxncon state, GSM controls, and Yeast9—rather than one narrowly structured generator?

This notebook is a readable research record. It follows the actual hand-offs in order and loads the saved evidence by default; it does **not** hide the experiment behind a one-cell runner.


## Pipeline at a glance

```text
goal → declared generator → observable/lockbox split → model setup & training
     → candidate or condition screen → matched comparison → interpretation
```

Each section below corresponds to one of these hand-offs.


In [ ]:
# Run this notebook from the repository root.
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

REGENERATE = False  # Cached artifacts are the default; no expensive solve runs implicitly.

def artifact(relative_path: str) -> Path:
    """Fail with a useful message rather than silently replacing evidence."""
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f"Missing cached artifact: {path}")
    return path

def show(frame, n=8):
    # `print` keeps this notebook usable in a plain Python kernel as well as Jupyter.
    print(frame.head(n).to_string(index=False))
    print(f"{len(frame):,} rows × {len(frame.columns):,} columns")


## 1. Experimental contract

The experiment has a declared observation boundary. “Observable” means the learner may use it; “lockbox” means it may be generated and audited but must not be used as a deployable feature.


In [ ]:
from yeast_validation import run_rxncon_process_variability_experiment as process

print("observable nominal inputs: temperature, pH, DO, time, reporters")
print("lockbox chain: realized process → rxncon state → GSM interface → bounds/fluxes")
print("world definitions:", [w.name for w in process.WORLDS])


## 2. Data generator

The nominal environment is observable. Realized process history drives rxncon/regulatory state, which produces the hidden GSM-interface controls used in the Yeast9 rollout. Bounds, fluxes, and hidden state remain lockboxed.

The next cell exposes the generator’s first concrete hand-off. It is deliberately small/inspection-only where generating the full campaign is expensive.


In [ ]:
# The named generator entry point exposes the causal hand-off in its return tables.
# The cached summary below is the primary evidence; do not launch the full campaign implicitly.
print("Process variability generator uses", process.N_TIMEPOINTS, "time points per culture.")


## 3. Model setup and training contract

Learning is evaluated only from the declared observable trajectories/reporters. The hidden rxncon-to-GSM interface is not made into a training feature merely because the generator can export it.

Training is not automatically started in this notebook. The cached training/evaluation artifacts below are the evidence record; regeneration must be an intentional, parameterized action.


In [ ]:
# Make the experiment hand-off inspectable before looking at aggregate metrics.
for name, relative_path in [('summary', 'data/rxncon_process_variability/process_variability_main_summary_interpreted.csv')]:
    path = artifact(relative_path)
    print(f"{name}: {path.relative_to(ROOT)}")


## 4. Screening / selection stage

Generator gates quantify interface complexity, process-variation effects, and dense-versus-sparse GSM rollout consistency before any performance claim.

The screen is intentionally shown separately from final verification, so a virtual score cannot be mistaken for an exact outcome.


In [ ]:
# Load the primary evidence table and inspect its schema before aggregation.
summary = pd.read_csv(artifact('data/rxncon_process_variability/process_variability_main_summary_interpreted.csv'))
show(summary)


## 5. Matched comparison

Compare baseline, process-variation, and shifted-world outcomes with their controls and failure modes visible.


In [ ]:
# Aggregate only over fields that exist in this version of the cached record.
comparison = summary.groupby('world', dropna=False).mean(numeric_only=True)
show(comparison.reset_index() if hasattr(comparison, "reset_index") else comparison)


## 6. Analysis view

The plot is intentionally generic: it exposes every numeric evidence column so the reader can select the metric relevant to the claim, rather than hard-coding an attractive subset.


In [ ]:
numeric = summary.select_dtypes("number")
if numeric.shape[1]:
    ax = numeric.plot(kind="box", rot=45, figsize=(11, 4), title="Cached evidence: numeric metric distribution")
    ax.set_ylabel("recorded metric value")
    plt.tight_layout()
else:
    print("This artifact has no numeric columns to plot.")


## 7. Interpretation, scope, and next hand-off

This widens structural validity. It is still synthetic and is not a claim that rxncon is a complete yeast regulatory ground truth.

### Reproduction boundary

The cells above reveal the inputs and artifacts without launching an expensive campaign. To regenerate, use the explicit command below only after reviewing its declared inputs and output destination.


In [ ]:
if REGENERATE:
    # This guard prevents accidental solver/campaign execution.
    raise RuntimeError('The rxncon/GSM campaign is expensive. Invoke its documented, parameterized generator only after choosing a world and output directory.')
